# Mini Project 2 — Part 2: Data Visualization

This notebook completes Part 2 using the `jmurph91_project_summary.csv` generated in Part 1.

For each assigned WoC project, this notebook:
- creates a complete monthly commit time series;
- includes zero-commit months;
- saves the monthly time series as a CSV;
- creates a 300-DPI line plot;
- finds the longest inactivity gap;
- counts inactivity gaps of at least three months;
- calculates commits after the longest gap;
- classifies the overall activity pattern; and
- creates the final project statistics CSV.

All generated per-project files are stored in the `part2_outputs` folder.

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NETID = "jmurph91"

# Part 1 input file
INPUT_FILE = f"{NETID}_project_summary.csv"

# Put all Part 2 output files together
OUTPUT_DIR = Path("part2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Input file: {INPUT_FILE}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

Input file: jmurph91_project_summary.csv
Output folder: /home/jmurph91/archeology/MiniProject2/MiniProject2/part2_outputs


In [27]:
# Load the Part 1 project summary CSV

df = pd.read_csv(
    INPUT_FILE,
    sep=";"
)

expected_columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

assert list(df.columns) == expected_columns, (
    f"Unexpected columns.\n"
    f"Expected: {expected_columns}\n"
    f"Found: {list(df.columns)}"
)

# Convert Unix timestamp to datetime
df["datetime"] = pd.to_datetime(
    df["time"],
    unit="s",
    errors="coerce"
)

assert df["datetime"].notna().all(), (
    "Some commit timestamps could not be converted."
)

# Create YYYY-MM month column
df["Month"] = df["datetime"].dt.to_period("M")

print(f"Loaded {len(df):,} commits.")
print(f"Projects: {df['project_wocid'].nunique()}")
print()
print(df["project_wocid"].value_counts().sort_index())

Loaded 50,009 commits.
Projects: 10

project_wocid
angel-ml_angel                         3418
bmclab_bmc                             2083
brial_brial                            2999
comit-network_xmr-btc-swap             4980
easystats_insight                      9905
jasp-stats_jasp-desktop               22023
juliarobotics_rigidbodydynamics.jl     2984
mmorise_world                           348
samreay_chainconsumer                   634
systempiper_systempipeshiny             635
Name: count, dtype: int64


In [28]:
def monthly_timeseries(project_df):
    """
    Create a monthly commit time series from the first
    month with activity through the last month with activity.
    
    Months with no commits are included with a value of 0.
    """
    
    counts = project_df.groupby("Month").size()

    first_month = counts.index.min()
    last_month = counts.index.max()

    # Every month between first and last commit
    all_months = pd.period_range(
        start=first_month,
        end=last_month,
        freq="M"
    )

    ts = (
        counts
        .reindex(all_months, fill_value=0)
        .rename_axis("Month")
        .reset_index(name="#Commits")
    )

    # Convert Period values to YYYY-MM strings
    ts["Month"] = ts["Month"].astype(str)

    ts["#Commits"] = ts["#Commits"].astype(int)

    return ts

In [29]:
def find_zero_gaps(ts):
    """
    Find every consecutive run of zero-commit months.
    """
    
    gaps = []
    start = None

    for i, row in ts.iterrows():

        month = row["Month"]
        is_zero = row["#Commits"] == 0

        # Beginning of a zero-commit period
        if is_zero and start is None:
            start = i

        # End of a zero-commit period
        if (not is_zero or i == len(ts) - 1) and start is not None:

            if is_zero and i == len(ts) - 1:
                end = i
            else:
                end = i - 1

            length = end - start + 1

            gaps.append({
                "start_index": start,
                "end_index": end,
                "start": ts.loc[start, "Month"],
                "end": ts.loc[end, "Month"],
                "length": length
            })

            start = None

    return gaps

In [30]:
def classify_activity(ts):
    """
    Classify the project's overall activity pattern.
    
    Categories:
    - steady
    - rising
    - declining
    - U-shaped
    - cyclical
    - irregular
    """

    values = ts["#Commits"].astype(float).to_numpy()

    n = len(values)

    if n < 3:
        return "irregular"

    # Linear trend
    x = np.arange(n)

    slope = np.polyfit(x, values, 1)[0]

    mean_value = values.mean()

    if mean_value == 0:
        return "irregular"

    normalized_slope = slope / mean_value

    # Divide timeline into thirds
    third = max(1, n // 3)

    early = values[:third].mean()
    middle = values[third:2 * third].mean()
    late = values[-third:].mean()

    # Avoid division problems
    if early == 0:
        early = 0.001

    # Coefficient of variation
    if mean_value != 0:
        cv = values.std() / mean_value
    else:
        cv = 0

    # Check for 12-month repetition
    lag12_corr = 0

    if n >= 36:
        first = values[:-12]
        second = values[12:]

        if np.std(first) > 0 and np.std(second) > 0:
            lag12_corr = np.corrcoef(first, second)[0, 1]

    # U-shaped pattern
    if (
        late > middle * 1.25
        and early > middle * 1.25
    ):
        return "U-shaped"

    # Cyclical / seasonal pattern
    if n >= 36 and lag12_corr >= 0.50:
        return "cyclical"

    # Rising trend
    if (
        normalized_slope >= 0.03
        and late > early * 1.20
    ):
        return "rising"

    # Declining trend
    if (
        normalized_slope <= -0.03
        and late < early * 0.80
    ):
        return "declining"

    # Steady activity
    if (
        abs(normalized_slope) < 0.03
        and cv < 1.0
    ):
        return "steady"

    # Anything else
    return "irregular"

In [ ]:
projects = sorted(df["project_wocid"].unique())

timeseries_results = {}

for project in projects:

    print(f"Processing {project}...")

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    # Build monthly time series
    ts = monthly_timeseries(project_df)

    timeseries_results[project] = ts

    # --------------------------------------------------------
    # Save monthly CSV
    # --------------------------------------------------------

    csv_path = (
        OUTPUT_DIR
        / f"{NETID}_commits_timeseries_{project}.csv"
    )

    ts.to_csv(
        csv_path,
        index=False,
        sep=";"
    )

    # --------------------------------------------------------
    # Create plot
    # --------------------------------------------------------

    plt.figure(figsize=(12, 6))

    plt.plot(
        ts["Month"],
        ts["#Commits"],
        marker="o",
        linestyle="-"
    )

    plt.xlabel("Month (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(project)

    plt.grid(True)

    # Don't make the x-axis unreadable for long projects
    if len(ts) > 60:
        step = max(1, len(ts) // 20)
        tick_positions = range(0, len(ts), step)

        plt.xticks(
            tick_positions,
            ts["Month"].iloc[list(tick_positions)],
            rotation=45,
            ha="right"
        )
    else:
        plt.xticks(rotation=45, ha="right")

    plt.tight_layout()

    plot_path = (
        OUTPUT_DIR
        / f"{NETID}_timeseries_{project}.png"
    )

    plt.savefig(
        plot_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"  {ts['Month'].iloc[0]} -> "
        f"{ts['Month'].iloc[-1]}"
    )

print()
print(f"Finished {len(projects)} projects.")
print(f"All files saved in: {OUTPUT_DIR.resolve()}")

Processing angel-ml_angel...
  2017-05 -> 2025-10
Processing bmclab_bmc...
  2013-11 -> 2025-11
Processing brial_brial...
  2006-03 -> 2025-10
Processing comit-network_xmr-btc-swap...
  2020-09 -> 2025-08
Processing easystats_insight...


In [ ]:
print("Files generated:")
print()

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)

In [ ]:
project_records = []

for project in projects:

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    ts = timeseries_results[project]

    # Find all zero-commit gaps
    gaps = find_zero_gaps(ts)

    # Only gaps of at least 3 months
    long_gaps = [
        gap for gap in gaps
        if gap["length"] >= 3
    ]

    # --------------------------------------------------------
    # Find longest inactivity gap
    # --------------------------------------------------------

    if gaps:

        # Longest gap
        # If tied, choose the earliest one
        longest_gap = sorted(
            gaps,
            key=lambda g: (-g["length"], g["start_index"])
        )[0]

        longest_gap_start = longest_gap["start"]
        longest_gap_end = longest_gap["end"]
        longest_gap_length = longest_gap["length"]

        # Number of commits after the end of the longest gap
        commits_after_gap = project_df[
            project_df["Month"]
            > pd.Period(longest_gap_end, freq="M")
        ].shape[0]

    else:

        longest_gap_start = ""
        longest_gap_end = ""
        longest_gap_length = 0
        commits_after_gap = 0

    # --------------------------------------------------------
    # Activity classification
    # --------------------------------------------------------

    activity_pattern = classify_activity(ts)

    # --------------------------------------------------------
    # GitHub information
    # --------------------------------------------------------

    github_row = github_info[
        github_info["project"] == project
    ]

    if len(github_row) == 1:

        github_row = github_row.iloc[0]

        stars = github_row["number of stars"]
        forks = github_row["number of forks"]
        last_commit_date = github_row["last commit date"]

    else:

        stars = np.nan
        forks = np.nan
        last_commit_date = ""

    # --------------------------------------------------------
    # Final project record
    # --------------------------------------------------------

    project_records.append({

        "project": project,

        "number of commits":
            len(project_df),

        "number of authors":
            project_df["author"].nunique(),

        "max time":
            project_df["time"].max(),

        "min time":
            project_df["time"].min(),

        "number of stars":
            stars,

        "number of forks":
            forks,

        "last commit date":
            last_commit_date,

        "LongestGapStart":
            longest_gap_start,

        "LongestGapEnd":
            longest_gap_end,

        "LongestGapLength":
            longest_gap_length,

        "NumCommitsAfterLastGap":
            commits_after_gap,

        "NumberOfGapsInTimeline":
            len(long_gaps),

        "ActivityPattern":
            activity_pattern
    })


project_stats = pd.DataFrame(project_records)

project_stats

In [ ]:
stats_file = (
    OUTPUT_DIR
    / f"{NETID}_project_stats.csv"
)

project_stats.to_csv(
    stats_file,
    index=False,
    sep=";"
)

print("Project statistics saved.")
print(f"File: {stats_file}")
print(f"Projects: {len(project_stats)}")

In [ ]:
expected_stats_columns = [
    "project",
    "number of commits",
    "number of authors",
    "max time",
    "min time",
    "number of stars",
    "number of forks",
    "last commit date",
    "LongestGapStart",
    "LongestGapEnd",
    "LongestGapLength",
    "NumCommitsAfterLastGap",
    "NumberOfGapsInTimeline",
    "ActivityPattern"
]

assert list(project_stats.columns) == expected_stats_columns

assert len(project_stats) == 10

assert project_stats["project"].nunique() == 10

print("✓ Final project statistics verified")
print(f"✓ Rows: {len(project_stats)}")
print(f"✓ Columns: {len(project_stats.columns)}")
print()
print(project_stats.to_string(index=False))

# Interpretation

The monthly commit time-series plots show how development activity changed over time for each project. The projects were classified as steady, rising, declining, U-shaped, cyclical, or irregular based on the overall patterns visible in their monthly activity.

A gap represents consecutive months with zero commits. The number of gaps reported below counts only gaps lasting at least three months.

In [ ]:
def interpretation_text(row):

    project = row["project"]
    pattern = row["ActivityPattern"]

    gap_count = int(
        row["NumberOfGapsInTimeline"]
    )

    gap_length = int(
        row["LongestGapLength"]
    )

    if gap_length > 0:

        gap_description = (
            f"The longest period of inactivity lasted "
            f"{gap_length} month(s), from "
            f"{row['LongestGapStart']} through "
            f"{row['LongestGapEnd']}."
        )

    else:

        gap_description = (
            "No zero-commit inactivity gaps were found."
        )

    return (
        f"{project}: The overall activity pattern is "
        f"{pattern}. "
        f"{gap_description} "
        f"There were {gap_count} inactivity gap(s) "
        f"lasting at least three months. "
        f"After the end of the longest inactivity gap, "
        f"the project recorded "
        f"{int(row['NumCommitsAfterLastGap']):,} commits."
    )


for _, row in project_stats.iterrows():

    print(interpretation_text(row))
    print()

In [ ]:
print("=" * 70)
print("PART 2 OUTPUT SUMMARY")
print("=" * 70)

print()
print(f"Input:")
print(f"  {INPUT_FILE}")

print()
print("Output folder:")
print(f"  {OUTPUT_DIR.resolve()}")

print()
print("Generated files:")

for file in sorted(OUTPUT_DIR.iterdir()):
    print(f"  ✓ {file.name}")

print()
print(f"Total output files: {len(list(OUTPUT_DIR.iterdir()))}")